# Phase 0: reproduce LLM-Check with a small reader on RAGTruth

Runs on a free **Kaggle T4** (Settings: Accelerator = GPU T4 x1, Internet = on) or **Colab T4**.

What this notebook does:
1. Clones the project branch and installs the few missing packages.
2. Logs in to Hugging Face (only needed for gated Llama / Gemma readers).
3. Scores 200 RAGTruth Summary answers with LLM-Check's original logit and attention scores, one forward pass each.
4. Shows the AUROC table and the attention-AUROC-by-layer plot.

Expected time: roughly 5 to 10 minutes for 200 answers with a 3B reader on a T4 (we record the exact seconds per answer).

In [ ]:
BRANCH = "claude/project-thread-ub5z89"   # change to "main" once the PR is merged
!git clone -q -b $BRANCH https://github.com/wolfykulfi/LLM_Check_Hallucination_Detection.git repo
%cd repo
!pip install -q "transformers>=4.45" accelerate pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Hugging Face token (gated models only)
Llama-3.2 and Gemma-2 need you to accept the licence on their model page and add a token:
* Kaggle: *Add-ons > Secrets*, name it `HF_TOKEN`.
* Colab: the key icon on the left, name it `HF_TOKEN`.

Qwen2.5 readers need no token: pick `configs/phase0_qwen25_3b.yaml` below instead.

In [ ]:
import os
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    from huggingface_hub import login
    login(token=token)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN found: use a Qwen config")

In [ ]:
CONFIG = "configs/phase0_llama32_3b.yaml"   # or configs/phase0_qwen25_3b.yaml / _1_5b / _0_5b
!python -m rag_halluc.repro.phase0_llmcheck --config $CONFIG

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display
out = sorted(glob.glob("results/phase0/*/auroc.csv"), key=os.path.getmtime)[-1]
table = pd.read_csv(out)
print(out)
display(table[~table.score.str.contains("_L")])
display(table[table.score.str.startswith("attn_L")].sort_values("auroc_best", ascending=False).head(5))
display(Image(out.replace("auroc.csv", "attn_auroc_by_layer.png")))

### Save the results
The CSVs are small. Download `results/phase0/` (Kaggle: *Output* tab; Colab: the file browser), or zip it:

In [ ]:
!zip -qr phase0_results.zip results/phase0 && ls -la phase0_results.zip